# Milestone 4: GRPO + LoRA debug run (Qwen3-1.7B) on a free Kaggle GPU

Trains Qwen3-1.7B with GRPO and a LoRA adapter on the real-filing **train** split, using the strict reward in
`xbrlbench/rl.py` (1 for a correct answer, 0.1 for a well-formed wrong one, 0 for scale errors and unparseable
answers). It then evaluates the trained model and the untrained base model on the 500-question held-out sample
through the same `xbrlbench run` path as milestone 3 (non-thinking mode).

**Setup (once):**
1. Kaggle → *Create* → *New notebook* → *File* → *Import notebook* → this file.
2. *Settings*: Accelerator **GPU T4 x2**, Internet **on** (phone-verified account).
   With two GPUs, vLLM generates on GPU 1 while training runs on GPU 0.
3. *Add input* → a dataset holding the **v3** corpus files `questions.jsonl` and `heldout_500.jsonl`
   (both from `data/filings/v3/`). Only the train split is used for training; the export refuses held-out questions.
4. *Save Version* → *Save & Run All (Commit)*. Progress streams into the version's *Logs*.
   Download `/kaggle/working/m4.zip` from *Output* at the end (it is also refreshed after each stage).

The defaults are a **20-step test run** (roughly 1-2 hours) that checks every stage works and shows whether the
reward moves. For the real run set `MAX_STEPS = 300`. No API keys are needed and nothing is billed.

In [ ]:
BRANCH = "claude/project-thread-aoax1i"   # branch with the M4 tooling (main once its PR merges)
MODEL = "Qwen/Qwen3-1.7B"
MAX_STEPS = 20            # test run; 300 for the real run
EVAL_STEPS = 10           # in-training validation every N steps (64 validation prompts, 2 samples each)
TRAIN_LIMIT = None        # e.g. 2000 prompts; None = the whole train split
EXCLUDE_FAMILIES = ""     # e.g. "gross_margin,current_ratio" if the base model already solves them
MAX_PROMPT_TOKENS = 3072
MAX_COMPLETION_TOKENS = 768
EVAL_BASE = True          # also evaluate the untrained model on the same v3 held-out file
EVAL_UNITS = True         # ...and the untrained model with the "units" prompt (the cheapest fix RL has to beat)
EVAL_LIMIT = None         # e.g. 100 for a quicker held-out check
WORKERS = 64
TIMEOUT = 3600            # seconds per evaluation request

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
rm -rf /kaggle/working/XBRLBench
git clone -q https://github.com/allisonzhan/XBRLBench /kaggle/working/XBRLBench

In [ ]:
import glob
import json
import os
import subprocess

REPO = "/kaggle/working/XBRLBench"
OUT = "/kaggle/working/m4"
subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
# One resolver pass so vLLM and the TRL pins agree on torch/transformers.
subprocess.run(["pip", "install", "-q", "vllm==0.31.0", "-e", f"{REPO}[train]"], check=True)
# Kaggle preinstalls torchao 0.10, which peft refuses (it needs >0.16) as soon as LoRA is
# applied. Nothing here uses torchao, so remove it.
subprocess.run(["pip", "uninstall", "-q", "-y", "torchao"], check=False)


def run_streaming(cmd, **kw):
    """Run cmd and print its output line by line, so progress shows up in a
    committed run's Logs (a plain subprocess.run writes past the notebook)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def zip_output():
    # Leave out the big rebuildable parts: merged weights, the fp16 base copy, checkpoints.
    subprocess.run(["bash", "-c", "cd /kaggle/working && rm -f m4.zip && zip -qr m4.zip m4 *.log "
                    "-x 'm4/run/merged/*' 'm4/run/base_fp16/*' 'm4/run/checkpoint-*'"], check=False)


def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert len(hits) == 1, f"need exactly one {name} in the inputs, found {hits or 'none'} (see setup step 3)"
    return hits[0]


QUESTIONS, HELDOUT = find("questions.jsonl"), find("heldout_500.jsonl")
corpus = {}
with open(QUESTIONS) as f:
    for line in f:
        q = json.loads(line)
        corpus[q["id"]] = q
held = [json.loads(line) for line in open(HELDOUT)]
# Both files must come from the same corpus build (v3), or the base-vs-trained comparison mixes wordings.
mismatch = [h["id"] for h in held if h["id"] not in corpus or corpus[h["id"]]["question"] != h["question"]]
assert not mismatch, f"{len(mismatch)} held-out questions differ from questions.jsonl (e.g. {mismatch[0]}): upload both from data/filings/v3"
assert all(h.get("split") == "test" for h in held)
print(f"corpus: {len(corpus)} questions, {sum(q.get('split') == 'train' for q in corpus.values())} train; held-out: {len(held)}")

TRAIN = f"{OUT}/data/train.jsonl"
cmd = ["python", "-m", "xbrlbench", "export-rl", "--in", QUESTIONS, "--out", TRAIN, "--val-fraction", "0.05"]
if EXCLUDE_FAMILIES:
    cmd += ["--exclude-families", EXCLUDE_FAMILIES]
run_streaming(cmd, cwd=REPO)
N_GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
assert N_GPUS >= 1, "No GPU: set Settings -> Accelerator to a GPU"
print("GPUs:", N_GPUS)

In [ ]:
import signal
import time
import urllib.request


def start(cmd, log, env=None, minutes=15):
    f = open(log, "w")
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, **(env or {})})
    for _ in range(minutes * 12):
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2)
            return proc
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError(open(log).read()[-3000:]) from None
            time.sleep(5)
    raise RuntimeError(f"server did not start in {minutes} minutes; see {log}")


def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    proc.wait(timeout=120)
    time.sleep(10)

## Train

Watch `rewards/accuracy_reward/mean` in the log lines: it should rise from about 0.2 (the base model's held-out accuracy in non-thinking mode).

In [ ]:
train_cmd = ["python", f"{REPO}/scripts/train_grpo.py", "--train", TRAIN, "--out", f"{OUT}/run",
             "--model", MODEL, "--max-steps", str(MAX_STEPS), "--eval-steps", str(EVAL_STEPS),
             "--save-steps", str(max(10, MAX_STEPS // 6)), "--merge",
             "--max-prompt-tokens", str(MAX_PROMPT_TOKENS), "--max-completion-tokens", str(MAX_COMPLETION_TOKENS)]
if TRAIN_LIMIT:
    train_cmd += ["--limit", str(TRAIN_LIMIT)]
if glob.glob(f"{OUT}/run/checkpoint-*"):
    train_cmd.append("--resume")

server = None
if N_GPUS >= 2:
    # vLLM generation server on GPU 1; TRL pushes the updated weights to it every step.
    server = start(["trl", "vllm-serve", "--model", MODEL, "--dtype", "half", "--gpu_memory_utilization", "0.9",
                    "--max_model_len", str(MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS)],
                   "/kaggle/working/vllm_train.log", env={"CUDA_VISIBLE_DEVICES": "1"})
    train_cmd += ["--vllm", "server"]
    train_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
else:
    train_cmd += ["--vllm", "colocate"]
    train_env = dict(os.environ)
t0 = time.time()
try:
    run_streaming(train_cmd, cwd=REPO, env=train_env)
finally:
    if server:
        stop(server)
    zip_output()
print(f"trained in {(time.time() - t0) / 60:.0f} min")

## Evaluate on the held-out sample

In [ ]:
def evaluate(model_path, runs):
    """runs: [(served name, --prompt variant)], all against one vLLM server."""
    proc = start(["vllm", "serve", model_path, "--served-model-name", *[n for n, _ in runs], "--dtype", "half",
                  "--tensor-parallel-size", str(N_GPUS), "--max-model-len", "8192",
                  "--gpu-memory-utilization", "0.90", "--port", "8000"],
                 f"/kaggle/working/vllm_{runs[0][0]}.log")
    try:
        for name, prompt in runs:
            cmd = ["python", "-m", "xbrlbench", "run", "--in", HELDOUT, "--api-base", "http://localhost:8000/v1",
                   "--models", name, "--reasoning", "off", "--prompt", prompt,
                   "--max-tokens", str(MAX_COMPLETION_TOKENS * 2),
                   "--workers", str(WORKERS), "--timeout", str(TIMEOUT), "--out-dir", f"{OUT}/eval/{name}"]
            if EVAL_LIMIT:
                cmd += ["--limit", str(EVAL_LIMIT)]
            if os.path.exists(f"{OUT}/eval/{name}/responses.jsonl"):
                cmd.append("--resume")
            run_streaming(cmd, cwd=REPO)
    finally:
        stop(proc)
        zip_output()

evaluate(f"{OUT}/run/merged", [("qwen3-1.7b-grpo", "default")])
base_runs = ([("qwen3-1.7b-nothink", "default")] if EVAL_BASE else []) + \
            ([("qwen3-1.7b-nothink-units", "units")] if EVAL_UNITS else [])
if base_runs:
    evaluate(MODEL, base_runs)

In [ ]:
import pathlib

rows = []
for p in sorted(pathlib.Path(f"{OUT}/eval").glob("*/responses.jsonl")):
    rows += p.read_text().splitlines(keepends=True)
pathlib.Path(f"{OUT}/eval/responses.jsonl").write_text("".join(rows))
run_streaming(["python", "-m", "xbrlbench", "report", "--in", f"{OUT}/eval/responses.jsonl",
               "--questions", HELDOUT, "--out-prefix", f"{OUT}/eval/report"], cwd=REPO)
run_streaming(["python", "-m", "xbrlbench", "reward", "--in", f"{OUT}/eval/responses.jsonl"], cwd=REPO)
print(open(f"{OUT}/eval/report.md").read())
zip_output()
print("Download /kaggle/working/m4.zip from the Output tab")